# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane:** Refresh Prediction (Lane 2 in the guide), reframed as a **weekly content-refresh budget allocation** problem—not a decline-classification exercise.

**Framing question:** *Given a limited weekly content-refresh budget, which pages should a FlyRank client refresh first to recover the most traffic?*

Clients do not have unlimited writer time. A binary “declining / not declining” label does not tell an editor what to do on Monday morning when they can only refresh *N* pages. The operational output is a **ranked queue**: expected traffic-recovery value per page, so scarce editor hours go to the highest-leverage URLs first. A classifier that is right on average but wrong on ordering still wastes the budget.

The value of a refresh is not only whether traffic is falling; it is **how much demand is on the table** (`impressions_90d`, position, CTR) and whether movement is plausible. In notebook 01, mean CTR falls sharply down the SERP—e.g. `top_3` ≈ 0.33% vs `page_3_5` ≈ 0.14% among pages with enough volume—so the **same editorial action** can mean very different click upside depending on current position tier. Prioritisation combines decline signal with visibility and position context; ML is one input to that score, not the decision by itself.

## 2. The question: decision, action, cost of a wrong call

**Decision:** For each client, rank every in-scope page by **expected traffic-recovery value if refreshed this week** (a priority score, not a yes/no decline label).

**Action:** A content editor takes the top *N* pages from that ranked list and performs a refresh (update, expand, or CTR/snippet fixes as appropriate).

**Unit of analysis:** One **page** (one `content_id` row), scoped **per client** (`client_id`).

**Cost of a wrong call:**
- **False positive (refresh too eagerly):** Editor capacity is spent on a page that would not have moved meaningfully—**hours of editor time** diverted from a better candidate. In this dataset that waste shows up as effort on rows with weak upside (e.g. low `impressions_90d` or a position tier where CTR upside is already thin).
- **False negative (refresh too late):** A declining page with real demand stays in the queue while **`impressions_90d` at risk** keep accruing without a refresh—organic visibility erodes without a surfaced recommendation.
- Ranking errors are asymmetric: missing a high-impression declining URL is usually costlier than over-reviewing a low-volume page, because the dataset ties stakes to measured exposure, not revenue.

## 3. Quick look at the data (2-3 real numbers)

Starter file: `data/raw/content_refresh_anonymized.csv` (plain pandas; no plots).

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

down_share = (df["trend_direction"] == "down").mean()
striking_ctr = df.loc[df["position_tier"] == "striking", "ctr"].mean()
top3_ctr = df.loc[df["position_tier"] == "top_3", "ctr"].mean()
striking_declining_share = (
    (df["position_tier"] == "striking") & (df["trend_direction"] == "down")
).mean()

print(f"Share of pages with trend_direction == 'down': {down_share:.1%}")
print(
    f"Mean ctr (striking tier): {striking_ctr:.4f}%  |  "
    f"mean ctr (top_3 tier): {top3_ctr:.4f}%"
)
print(f"Share of pages both striking tier and declining: {striking_declining_share:.1%}")


Share of pages with trend_direction == 'down': 54.2%
Mean ctr (striking tier): 0.3232%  |  mean ctr (top_3 tier): 1.4836%
Share of pages both striking tier and declining: 14.8%


- **Decline base rate (~54%):** More than half of rows are already tagged `down` in the trailing window—there is plenty of candidate volume, so the capstone problem is *ordering*, not finding a rare class.
- **CTR gap (top_3 vs striking on mean `ctr`):** Higher slots earn materially higher measured CTR in this slice (`ctr` is stored as a percent, e.g. 1.48 means 1.48%), so position tier belongs in the recovery-value score—not a separate “CTR model.”
- **Striking + declining (~15% of inventory):** That intersection is a large, concrete refresh pool where decline signal and page-one-adjacent visibility overlap—the population this lane is meant to prioritise first.

## 4. Careful words: what I can and can't claim

- In this dataset I can describe **associations** between observable signals (`impressions_90d`, `ctr`, `position_tier`, `trend_direction`) and which pages look like strong refresh candidates.
- I can support **decision-support** ranking: “review these pages first given limited weekly capacity,” with reason codes tied to columns we can inspect.
- I can report **directional** patterns (e.g. higher mean CTR in `top_3` than in `striking` here) using careful, within-sample language.
- I **cannot** claim a refresh **caused** traffic to return without an experiment or causal design—only that prioritisation is consistent with observed demand and movement.
- I **cannot** claim algorithm insight, guaranteed rankings, or outcomes for pages outside this anonymised sample.
- I **cannot** translate impressions into revenue; stakes stay in **editor time** and **`impressions_90d` at risk**, as recorded.

## Self-check

- **Actionable this week?** Yes—an editor can take the top *N* ranked pages and refresh them; the output is a queue, not a research memo.
- **Cost quantified?** Yes—false positives cost editor time; false negatives leave measurable `impressions_90d` on declining pages (no dollar fiction).
- **Columns in hand?** Mostly: grain, demand, position, CTR, and a proxy decline flag exist; a **forward-window recovery label** and explicit **editor-hour** field are not in the starter CSV (warehouse/time split work comes later).
- **Useful if the model is mediocre?** Yes—a transparent baseline rank plus clear reason codes still beats an unordered inventory; ML only needs to improve ordering at top-*K*.

Submission checklist:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.